In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [17]:
# =========================
# 1) Imports
# =========================
import numpy as np
import pandas as pd
import gdown
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K
from tensorflow.keras import initializers, regularizers, constraints
import tensorflow as tf

In [18]:
# =========================
# 2) Load data and rebuild the SAME split as training notebook
# =========================
url = "https://drive.google.com/file/d/1gND_ylSniEH7F7BTqEpS6QEqdg78Clkh/view?usp=sharing"
output_filename = 'df.pkl'
file_path = gdown.download(url, output_filename, quiet=False, fuzzy=True)
df = pd.read_pickle(file_path)

identity_columns = [
    'male', 'female', 'homosexual_gay_or_lesbian', 'christian', 'jewish',
    'muslim', 'black', 'white', 'psychiatric_or_mental_illness'
]

# Match the preprocessing in the training notebook
for column in identity_columns + ['target']:
    df[column] = np.where(df[column] >= 0.5, True, False)

TEXT_COLUMN = 'comment_text'
TOXICITY_COLUMN = 'target'

y = df[TOXICITY_COLUMN].astype(int).values
train_df, cv_df, y_train, y_cv = train_test_split(
    df, y, test_size=0.2, random_state=42
)

# validation labels for classification_report
validate_y = to_categorical(cv_df[TOXICITY_COLUMN].astype(int))

Downloading...
From (original): https://drive.google.com/uc?id=1gND_ylSniEH7F7BTqEpS6QEqdg78Clkh
From (redirected): https://drive.google.com/uc?id=1gND_ylSniEH7F7BTqEpS6QEqdg78Clkh&confirm=t&uuid=8eebd8d3-0b05-44ca-a5f3-8bb8319afab6
To: /kaggle/working/df.pkl
100%|██████████| 1.22G/1.22G [00:11<00:00, 104MB/s] 


In [19]:
# =========================
# 3) Load tokenizer and pad validation text
# =========================
MAX_SEQUENCE_LENGTH = 300

url = "https://drive.google.com/file/d/15PWNsDqb-QFg34YQ90GUGnO9gi1YQEgs/view?usp=sharing"
output_filename = 'tokenizer.pkl'
file_path = gdown.download(url, output_filename, quiet=False, fuzzy=True)
tokenizer = pd.read_pickle(file_path)

def padding_text(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_SEQUENCE_LENGTH
    )

validate_text = padding_text(cv_df[TEXT_COLUMN])

Downloading...
From: https://drive.google.com/uc?id=15PWNsDqb-QFg34YQ90GUGnO9gi1YQEgs
To: /kaggle/working/tokenizer.pkl
100%|██████████| 12.7M/12.7M [00:00<00:00, 20.4MB/s]


In [20]:
# =========================
# 4) Copy the metric functions from the training notebook
# =========================
SUBGROUP_AUC = 'subgroup_auc'
BPSN_AUC = 'bpsn_auc'
BNSP_AUC = 'bnsp_auc'

def compute_auc(y_true, y_pred):
    try:
        from sklearn import metrics
        return metrics.roc_auc_score(y_true, y_pred)
    except ValueError:
        return np.nan

def compute_subgroup_auc(df, subgroup, label, model_name):
    subgroup_examples = df[df[subgroup]]
    return compute_auc(subgroup_examples[label], subgroup_examples[model_name])

def compute_bpsn_auc(df, subgroup, label, model_name):
    subgroup_negative_examples = df[df[subgroup] & ~df[label]]
    non_subgroup_positive_examples = df[~df[subgroup] & df[label]]
    examples = pd.concat([subgroup_negative_examples, non_subgroup_positive_examples])
    return compute_auc(examples[label], examples[model_name])

def compute_bnsp_auc(df, subgroup, label, model_name):
    subgroup_positive_examples = df[df[subgroup] & df[label]]
    non_subgroup_negative_examples = df[~df[subgroup] & ~df[label]]
    examples = pd.concat([subgroup_positive_examples, non_subgroup_negative_examples])
    return compute_auc(examples[label], examples[model_name])

def compute_bias_metrics_for_model(dataset, subgroups, model, label_col):
    records = []
    for subgroup in subgroups:
        record = {
            'subgroup': subgroup,
            'subgroup_size': len(dataset[dataset[subgroup]])
        }
        record[SUBGROUP_AUC] = compute_subgroup_auc(dataset, subgroup, label_col, model)
        record[BPSN_AUC] = compute_bpsn_auc(dataset, subgroup, label_col, model)
        record[BNSP_AUC] = compute_bnsp_auc(dataset, subgroup, label_col, model)
        records.append(record)
    return pd.DataFrame(records).sort_values('subgroup_auc', ascending=True)

def calculate_overall_auc(df, model_name):
    true_labels = df[TOXICITY_COLUMN]
    predicted_labels = df[model_name]
    return compute_auc(true_labels, predicted_labels)

def power_mean(series, p):
    total = sum(np.power(series, p))
    return np.power(total / len(series), 1 / p)

def get_final_metric(bias_df, overall_auc, POWER=-5, OVERALL_MODEL_WEIGHT=0.25):
    bias_score = np.average([
        power_mean(bias_df[SUBGROUP_AUC], POWER),
        power_mean(bias_df[BPSN_AUC], POWER),
        power_mean(bias_df[BNSP_AUC], POWER)
    ])
    return (OVERALL_MODEL_WEIGHT * overall_auc) + ((1 - OVERALL_MODEL_WEIGHT) * bias_score)

In [21]:
# =========================
# 6) Load already-trained models
# =========================
LRModel = pd.read_pickle(gdown.download(
    "https://drive.google.com/file/d/1A40POUq1g1EqSlE76zqqyHo4KqzLVtub/view?usp=sharing",
    "LogisticRegressionModel.pkl",
    quiet=False, fuzzy=True
))

LV = pd.read_pickle(gdown.download(
    "https://drive.google.com/file/d/1BhsF97-CgeZj5UDT5WozXC564KtaQ2WK/view?usp=sharing",
    "LogisticVectorizer.pkl",
    quiet=False, fuzzy=True
))

Downloading...
From: https://drive.google.com/uc?id=1A40POUq1g1EqSlE76zqqyHo4KqzLVtub
To: /kaggle/working/LogisticRegressionModel.pkl
100%|██████████| 97.4M/97.4M [00:00<00:00, 126MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1BhsF97-CgeZj5UDT5WozXC564KtaQ2WK
To: /kaggle/working/LogisticVectorizer.pkl
100%|██████████| 56.4M/56.4M [00:00<00:00, 119MB/s] 


In [26]:
SingleLSTM_model = load_model('/kaggle/input/models/debangshudey2004/singlelstm/keras/default/1/SingleLSTM_Model.keras')
BidirectionalLSTM_model = load_model('/kaggle/input/models/debangshudey2004/bidirectionallstm/keras/default/1/BidirectionalLSTM_Model.keras')
WSBLSTM_model = load_model('/kaggle/input/models/debangshudey2004/weightedsamplesbilstm/keras/default/1/WeightedSamplesBidirectionalLSTM_Model.keras')

In [27]:
# =========================
# 7) Generic evaluation helpers
# =========================
def evaluate_binary_predictions(y_true, y_score, y_pred, model_name, eval_df):
    print(f"\n================ {model_name} ================")
    print(classification_report(y_true, y_pred, digits=4))

    temp_df = eval_df.copy()
    temp_df[model_name] = y_score

    bias_metrics_df = compute_bias_metrics_for_model(
        temp_df, identity_columns, model_name, TOXICITY_COLUMN
    )
    final_metric = get_final_metric(
        bias_metrics_df,
        calculate_overall_auc(temp_df, model_name)
    )

    print(f"Final metric ({model_name}): {final_metric:.6f}")
    return bias_metrics_df, final_metric

def evaluate_sklearn_model(model, X_eval, y_true, model_name, eval_df):
    y_score = model.predict_proba(X_eval)[:, 1]
    y_pred = (y_score >= 0.5).astype(int)
    return evaluate_binary_predictions(y_true, y_score, y_pred, model_name, eval_df)

def evaluate_keras_model(model, X_eval, y_true, y_onehot, model_name, eval_df):
    y_prob = model.predict(X_eval, verbose=0)

    if y_prob.ndim == 2 and y_prob.shape[1] > 1:
        y_score = y_prob[:, 1]
        y_pred = np.argmax(y_prob, axis=1)
    else:
        y_score = y_prob.ravel()
        y_pred = (y_score >= 0.5).astype(int)

    y_true_cls = np.argmax(y_onehot, axis=1)
    return evaluate_binary_predictions(y_true_cls, y_score, y_pred, model_name, eval_df)

In [28]:
# =========================
# 8) Run all models without retraining
# =========================
results = {}

# Logistic Regression
results["logreg"] = evaluate_sklearn_model(
    LRModel,
    LV.transform(cv_df[TEXT_COLUMN]),
    cv_df[TOXICITY_COLUMN].astype(int).values,
    "logreg_model",
    cv_df
)

# Single LSTM
results["single_lstm"] = evaluate_keras_model(
    SingleLSTM_model,
    validate_text,
    cv_df[TOXICITY_COLUMN].astype(int).values,
    validate_y,
    "slstm_model",
    cv_df
)

# BiLSTM
results["bilstm"] = evaluate_keras_model(
    BidirectionalLSTM_model,
    validate_text,
    cv_df[TOXICITY_COLUMN].astype(int).values,
    validate_y,
    "blstm_model",
    cv_df
)

# Weighted BiLSTM
results["weighted_bilstm"] = evaluate_keras_model(
    WSBLSTM_model,
    validate_text,
    cv_df[TOXICITY_COLUMN].astype(int).values,
    validate_y,
    "blstm_model_weighted",
    cv_df
)


================ logreg_model ================
              precision    recall  f1-score   support

           0     0.9536    0.9853    0.9692    332165
           1     0.7250    0.4477    0.5535     28810

    accuracy                         0.9424    360975
   macro avg     0.8393    0.7165    0.7614    360975
weighted avg     0.9354    0.9424    0.9360    360975

Final metric (logreg_model): 0.867870


I0000 00:00:1780594135.947616     204 cuda_dnn.cc:529] Loaded cuDNN version 91002



================ slstm_model ================
              precision    recall  f1-score   support

           0     0.9541    0.9897    0.9716    332165
           1     0.7920    0.4516    0.5753     28810

    accuracy                         0.9468    360975
   macro avg     0.8731    0.7207    0.7734    360975
weighted avg     0.9412    0.9468    0.9400    360975

Final metric (slstm_model): 0.892696

================ blstm_model ================
              precision    recall  f1-score   support

           0     0.9908    0.8025    0.8868    332165
           1     0.2865    0.9140    0.4362     28810

    accuracy                         0.8114    360975
   macro avg     0.6386    0.8582    0.6615    360975
weighted avg     0.9346    0.8114    0.8508    360975

Final metric (blstm_model): 0.899173

================ blstm_model_weighted ================
              precision    recall  f1-score   support

           0     0.9880    0.8541    0.9161    332165
           1 